In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_1.py ──
"""
Shared infrastructure for MLFP02 Exercise 1 — Probability and Bayesian
Fundamentals.

Contains: HDB data loading (4-ROOM 2020+ slice), prior/posterior math for
the Normal-Normal and Beta-Binomial conjugate families, bootstrap helpers,
and output directory wiring.

Technique-specific narrative (MLE derivation, Bayes scenarios, credible vs
confidence simulation, bootstrap comparison) does NOT belong here — it
lives in the per-technique files.
"""

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import polars as pl
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

# Output directory for plots (HTML) — every technique writes here.
OUTPUT_DIR = Path("outputs") / "mlfp02_ex1_bayes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Canonical prior used across all four technique files. A single source
# of truth means Task 5's prior sweep and Task 4's posterior use the same
# starting point.
PRIOR_MU_0: float = 500_000.0  # SGD — Singapore 4-room HDB market anchor
PRIOR_SIGMA_0: float = 100_000.0  # SGD — moderate uncertainty

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale flats (Singapore, data.gov.sg)
# ════════════════════════════════════════════════════════════════════════

_loader = MLFPDataLoader()


def load_hdb_all() -> pl.DataFrame:
    """Full HDB resale dataset filtered to 2020+ transactions.

    Returns a polars DataFrame with columns: month, town, flat_type,
    resale_price, plus any others present in the source parquet. Used
    by Task 1 (truth tables) and Task 8 (expected value by flat type).
    """
    hdb = _loader.load("mlfp01", "hdb_resale.parquet")
    return hdb.filter(pl.col("month").str.to_date("%Y-%m") >= pl.date(2020, 1, 1))


def load_hdb_4room() -> pl.DataFrame:
    """4-ROOM slice of HDB resale (2020+) — the primary estimation target."""
    return load_hdb_all().filter(pl.col("flat_type") == "4 ROOM")


def load_hdb_prices_4room() -> np.ndarray:
    """Return the 4-room resale_price column as a float64 numpy array.

    This is the primary observation vector for MLE, Normal-Normal, and
    bootstrap tasks.
    """
    return load_hdb_4room()["resale_price"].to_numpy().astype(np.float64)


# ════════════════════════════════════════════════════════════════════════
# NORMAL-NORMAL CONJUGATE
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class NormalPosterior:
    """Posterior for μ under a Normal-Normal conjugate with known σ."""

    mean: float
    std: float
    prior_mean: float
    prior_std: float
    n: int
    sigma_known: float

    @property
    def precision_prior(self) -> float:
        return 1.0 / self.prior_std**2

    @property
    def precision_data(self) -> float:
        return self.n / self.sigma_known**2

    @property
    def prior_weight(self) -> float:
        """Fraction of posterior precision contributed by the prior (0..1)."""
        return self.precision_prior / (self.precision_prior + self.precision_data)

    def credible_interval(self, level: float = 0.95) -> tuple[float, float]:
        z = stats.norm.ppf(0.5 + level / 2)
        return (self.mean - z * self.std, self.mean + z * self.std)


def normal_normal_posterior(
    data: np.ndarray,
    prior_mean: float,
    prior_std: float,
    sigma_known: float,
) -> NormalPosterior:
    """Closed-form posterior for μ under N(μ₀, σ₀²) prior and known σ.

    Posterior precision = prior precision + n / σ². Posterior mean is the
    precision-weighted average of the prior mean and the sample mean.
    """
    n = len(data)
    xbar = float(np.mean(data))
    prec_prior = 1.0 / prior_std**2
    prec_data = n / sigma_known**2
    prec_post = prec_prior + prec_data
    sigma_post_sq = 1.0 / prec_post
    mu_post = sigma_post_sq * (prior_mean * prec_prior + n * xbar / sigma_known**2)
    return NormalPosterior(
        mean=mu_post,
        std=float(np.sqrt(sigma_post_sq)),
        prior_mean=prior_mean,
        prior_std=prior_std,
        n=n,
        sigma_known=sigma_known,
    )


# ════════════════════════════════════════════════════════════════════════
# BETA-BINOMIAL CONJUGATE
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class BetaPosterior:
    """Posterior for p under a Beta-Binomial conjugate."""

    alpha: float
    beta: float
    prior_alpha: float
    prior_beta: float
    k: int
    n: int

    @property
    def mean(self) -> float:
        return self.alpha / (self.alpha + self.beta)

    @property
    def prior_mean(self) -> float:
        return self.prior_alpha / (self.prior_alpha + self.prior_beta)

    def credible_interval(self, level: float = 0.95) -> tuple[float, float]:
        lo = (1 - level) / 2
        hi = 1 - lo
        return tuple(stats.beta.ppf([lo, hi], self.alpha, self.beta).tolist())


def beta_binomial_posterior(
    k: int, n: int, prior_alpha: float, prior_beta: float
) -> BetaPosterior:
    """Closed-form posterior for p under Beta(α, β) prior and k/n Binomial."""
    return BetaPosterior(
        alpha=prior_alpha + k,
        beta=prior_beta + (n - k),
        prior_alpha=prior_alpha,
        prior_beta=prior_beta,
        k=k,
        n=n,
    )


# ════════════════════════════════════════════════════════════════════════
# MLE + CRAMER-RAO
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class NormalMLE:
    mean: float
    mle_std: float  # ddof=0, biased
    unbiased_std: float  # ddof=1, Bessel's correction
    n: int

    @property
    def fisher_information(self) -> float:
        return self.n / self.mle_std**2

    @property
    def cramer_rao_bound(self) -> float:
        return 1.0 / self.fisher_information

    @property
    def standard_error(self) -> float:
        return float(np.sqrt(self.cramer_rao_bound))


def normal_mle(data: np.ndarray) -> NormalMLE:
    """MLE for Normal (μ, σ²) with Cramér-Rao bound bookkeeping."""
    arr = np.asarray(data, dtype=np.float64)
    return NormalMLE(
        mean=float(arr.mean()),
        mle_std=float(arr.std(ddof=0)),
        unbiased_std=float(arr.std(ddof=1)),
        n=len(arr),
    )


# ════════════════════════════════════════════════════════════════════════
# BOOTSTRAP INTERVALS
# ════════════════════════════════════════════════════════════════════════


def bootstrap_mean_distribution(
    data: np.ndarray, n_bootstrap: int = 10_000, seed: int = 42
) -> np.ndarray:
    """Non-parametric bootstrap of the sample mean."""
    rng = np.random.default_rng(seed)
    n = len(data)
    return np.array(
        [rng.choice(data, size=n, replace=True).mean() for _ in range(n_bootstrap)]
    )


def percentile_ci(draws: np.ndarray, level: float = 0.95) -> tuple[float, float]:
    lo = (1 - level) / 2 * 100
    hi = (1 + level) / 2 * 100
    return float(np.percentile(draws, lo)), float(np.percentile(draws, hi))


def bca_ci(
    data: np.ndarray, n_bootstrap: int = 10_000, seed: int = 42, level: float = 0.95
) -> tuple[float, float]:
    """Bias-corrected accelerated bootstrap CI for the mean (via scipy)."""
    result = stats.bootstrap(
        (np.asarray(data, dtype=np.float64),),
        statistic=np.mean,
        n_resamples=n_bootstrap,
        confidence_level=level,
        method="BCa",
        random_state=seed,
    )
    return float(result.confidence_interval.low), float(result.confidence_interval.high)


# ════════════════════════════════════════════════════════════════════════
# FORMATTING
# ════════════════════════════════════════════════════════════════════════


def fmt_money(x: float) -> str:
    return f"${x:,.0f}"


def print_interval(label: str, lo: float, hi: float) -> None:
    print(
        f"  {label:<28} [{fmt_money(lo)}, {fmt_money(hi)}]  width={fmt_money(hi - lo)}"
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 1.4: Intervals — Credible vs Confidence, Bootstrap,
#                         and Flat-Type Comparison
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Distinguish Bayesian credible intervals from frequentist confidence
#     intervals with a repeated-sampling simulation
#   - Compute bootstrap CIs (percentile + BCa) and compare with theory
#   - Apply Bayesian estimation across flat types to see how prior
#     influence varies with sample size (Bayesian regularisation)
#   - Understand expected value and sampling bias (friendship paradox)
#   - Synthesize all findings into a stakeholder-ready business summary
#
# PREREQUISITES: Complete 03_conjugate_priors.py (Normal-Normal, Beta-Binomial)
#
# ESTIMATED TIME: ~50 min
#
# TASKS:
#   1. Theory — credible vs confidence: the philosophical difference
#   2. Build — repeated-sampling simulation (1,000 experiments)
#   3. Train — bootstrap CIs + expected value + sampling bias
#   4. Visualise — interval comparison + flat-type posterior chart
#   5. Apply — Bayesian regularisation across flat types + business synthesis
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
import polars as pl
from kailash_ml import ModelVisualizer
from plotly.subplots import make_subplots
from scipy import stats



## THEORY — Credible vs Confidence: The Philosophical Difference


In [ ]:
# Frequentist confidence interval (CI):
#   "If I repeated this experiment many times, 95% of the intervals
#    I construct would contain the true μ."
#   → Statement about the PROCEDURE, not about THIS interval.
#
# Bayesian credible interval:
#   "Given THIS data, there is a 95% probability that μ lies in
#    this interval."
#   → Statement about THIS interval, not about a procedure.
#
# Which is "better"? Neither — they answer different questions. But the
# Bayesian interpretation is what most people THINK the frequentist CI
# means.
#
# Bootstrap CIs are a frequentist method that relaxes distributional
# assumptions. BCa (bias-corrected accelerated) corrects for both bias
# and skewness in the bootstrap distribution.



## TASK 2 — BUILD: Repeated-Sampling Simulation


In [ ]:
print("\n" + "=" * 70)
print("  MLFP02 Exercise 1.4: Intervals and Flat-Type Comparison")
print("=" * 70)

prices = load_hdb_prices_4room()
mle = normal_mle(prices)
mu_0 = PRIOR_MU_0
sigma_0 = PRIOR_SIGMA_0

rng = np.random.default_rng(seed=42)
true_mu = mle.mean  # Treat our MLE as the "true" population mean
true_sigma = mle.mle_std
n_simulations = 1000
sample_size = 100  # Small sample to show variation

freq_covers = 0
bayes_covers = 0
freq_widths = []
bayes_widths = []

print(f"\n=== Credible vs Confidence Interval Simulation ===")
print(f"Simulations: {n_simulations:,}, sample size: {sample_size}")

for _ in range(n_simulations):
    sample = rng.normal(true_mu, true_sigma, size=sample_size)
    xbar = sample.mean()
    se = sample.std(ddof=1) / np.sqrt(sample_size)

    # TODO: Compute frequentist 95% CI using xbar ± 1.96 * se
    freq_lower = ____
    freq_upper = ____
    if freq_lower <= true_mu <= freq_upper:
        freq_covers += 1
    freq_widths.append(freq_upper - freq_lower)

    # TODO: Compute Bayesian 95% credible interval using normal_normal_posterior
    # Hint: normal_normal_posterior(sample, mu_0, sigma_0, sample.std(ddof=0))
    post_sim = ____
    bayes_lower, bayes_upper = post_sim.credible_interval(0.95)
    if bayes_lower <= true_mu <= bayes_upper:
        bayes_covers += 1
    bayes_widths.append(bayes_upper - bayes_lower)

freq_coverage = freq_covers / n_simulations
bayes_coverage = bayes_covers / n_simulations

print(f"Frequentist CI coverage: {freq_coverage:.1%} (target: 95%)")
print(f"Bayesian credible coverage: {bayes_coverage:.1%}")
print(f"Mean freq CI width: {fmt_money(np.mean(freq_widths))}")
print(f"Mean Bayes CI width: {fmt_money(np.mean(bayes_widths))}")
# INTERPRETATION: Both methods achieve approximately 95% coverage when
# the model is correct. The key difference is philosophical: the freq CI
# is about the procedure, the Bayes CI is about THIS interval.



### Checkpoint 1


In [ ]:
assert (
    0.90 < freq_coverage < 1.0
), f"Frequentist coverage should be near 95%, got {freq_coverage:.1%}"
assert (
    0.90 < bayes_coverage < 1.0
), f"Bayesian coverage should be near 95%, got {bayes_coverage:.1%}"
print("\n✓ Checkpoint 1 passed — coverage simulation validates both methods\n")



## TASK 3 — TRAIN: Bootstrap CIs + Expected Value + Sampling Bias


In [ ]:
# --- Bootstrap confidence intervals ---
n_bootstrap = 10_000

# TODO: Generate bootstrap mean distribution and compute percentile CI
# Hint: bootstrap_mean_distribution(prices, n_bootstrap=n_bootstrap, seed=42)
bootstrap_means = ____
boot_lo, boot_hi = percentile_ci(bootstrap_means, level=0.95)

# TODO: Compute BCa CI using the shared helper
# Hint: bca_ci(prices, n_bootstrap=n_bootstrap, seed=42, level=0.95)
bca_lo, bca_hi = ____

# Normal theory CI for reference
normal_ci_lo = mle.mean - 1.96 * mle.standard_error
normal_ci_hi = mle.mean + 1.96 * mle.standard_error

# Bayesian CI from full dataset
posterior_full = normal_normal_posterior(prices, mu_0, sigma_0, mle.mle_std)
bayes_ci_lo, bayes_ci_hi = posterior_full.credible_interval(0.95)

print("=== Confidence / Credible Intervals Comparison ===")
print(f"{'Method':<25} {'Lower':>14} {'Upper':>14} {'Width':>12}")
print("─" * 70)
print_interval("Normal theory 95% CI", normal_ci_lo, normal_ci_hi)
print_interval("Bootstrap percentile CI", boot_lo, boot_hi)
print_interval("Bootstrap BCa CI", bca_lo, bca_hi)
print_interval("Bayesian 95% credible", bayes_ci_lo, bayes_ci_hi)

# --- Expected value by flat type ---
print(f"\n=== Expected Value: HDB Price by Flat Type ===")
hdb_all = load_hdb_all()
flat_type_stats = (
    hdb_all.group_by("flat_type")
    .agg(
        pl.col("resale_price").mean().alias("mean_price"),
        pl.len().alias("count"),
    )
    .sort("flat_type")
)
total_transactions = flat_type_stats["count"].sum()
flat_type_stats = flat_type_stats.with_columns(
    (pl.col("count") / total_transactions).alias("probability")
)
print(flat_type_stats)

# TODO: Compute E[price] = Σ P(flat_type) × E[price | flat_type]
# Hint: multiply probability column by mean_price column, then .sum()
expected_price = ____
simple_avg = flat_type_stats["mean_price"].mean()
print(f"\nE[price] (transaction-weighted) = {fmt_money(expected_price)}")
print(f"Simple average across types     = {fmt_money(simple_avg)}")
print(f"Difference: {fmt_money(expected_price - simple_avg)}")

# --- Friendship paradox (sampling bias) ---
print(f"\n=== Sampling Bias: Friendship Paradox ===")
n_people = 200
degrees = rng.zipf(a=2.0, size=n_people).clip(max=n_people - 1)
avg_degree = degrees.mean()

friend_degrees = []
for person_idx in range(n_people):
    if degrees[person_idx] > 0:
        # A friend is drawn proportional to degree (popularity)
        friend_probs = degrees / degrees.sum()
        friend_idx = rng.choice(n_people, p=friend_probs)
        friend_degrees.append(degrees[friend_idx])

avg_friend_degree = np.mean(friend_degrees)

print(f"People: {n_people}")
print(f"Your average number of friends: {avg_degree:.1f}")
print(f"Your friends' average number of friends: {avg_friend_degree:.1f}")
print(f"Ratio: {avg_friend_degree / avg_degree:.2f}x")
print(
    f"→ Your friends have {avg_friend_degree / avg_degree:.1f}x more friends than you!"
)



### Checkpoint 2


In [ ]:
assert boot_lo < boot_hi, "Bootstrap CI lower must be below upper"
assert bca_lo < bca_hi, "BCa CI lower must be below upper"
assert avg_friend_degree > avg_degree, "Friends should have more friends (paradox)"
assert expected_price > 0, "Expected price must be positive"
print("\n✓ Checkpoint 2 passed — bootstrap, expected value, sampling bias computed\n")



## TASK 4 — VISUALISE: Interval Comparison + Flat-Type Posteriors


In [ ]:
viz = ModelVisualizer()

# -- Plot 1: Bootstrap distribution histogram --
fig1 = go.Figure()
fig1.add_trace(
    go.Histogram(
        x=bootstrap_means,
        nbinsx=80,
        name="Bootstrap means",
        marker_color="steelblue",
        opacity=0.7,
    )
)
fig1.add_vline(
    x=mle.mean,
    line_dash="solid",
    line_color="red",
    annotation_text=f"MLE: {fmt_money(mle.mean)}",
)
fig1.add_vline(x=boot_lo, line_dash="dash", line_color="green", annotation_text="2.5%")
fig1.add_vline(x=boot_hi, line_dash="dash", line_color="green", annotation_text="97.5%")
fig1.update_layout(
    title="Bootstrap Distribution of Sample Mean (10,000 resamples)",
    xaxis_title="Mean Price ($)",
    yaxis_title="Count",
    height=400,
)
fig1.write_html(str(OUTPUT_DIR / "bootstrap_distribution.html"))
print("Saved: bootstrap_distribution.html")

# -- Plot 2: Bayesian estimates across flat types --
flat_types = ["2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE"]
results_by_type = {}
hdb_full = load_hdb_all()

for ft in flat_types:
    subset = hdb_full.filter(pl.col("flat_type") == ft)
    if subset.height == 0:
        continue
    p = subset["resale_price"].to_numpy().astype(np.float64)
    # TODO: Compute posterior for each flat type
    # Hint: normal_normal_posterior(p, mu_0, sigma_0, p.std())
    post_ft = ____
    ci_ft = post_ft.credible_interval(0.95)
    results_by_type[ft] = {
        "n": len(p),
        "mle_mean": float(p.mean()),
        "posterior_mean": post_ft.mean,
        "posterior_std": post_ft.std,
        "prior_weight": post_ft.prior_weight * 100,
        "ci_lower": ci_ft[0],
        "ci_upper": ci_ft[1],
    }

print(f"\n=== Bayesian Estimates by Flat Type ===")
print(
    f"{'Type':<12} {'n':>8} {'MLE Mean':>12} {'Post Mean':>12} {'Post σ':>10} {'Prior %':>8}"
)
print("─" * 70)
for ft, r in results_by_type.items():
    print(
        f"{ft:<12} {r['n']:>8,} ${r['mle_mean']:>10,.0f} "
        f"${r['posterior_mean']:>10,.0f} ${r['posterior_std']:>8,.2f} "
        f"{r['prior_weight']:>7.3f}%"
    )

# Flat-type comparison scatter
if results_by_type:
    ft_names = list(results_by_type.keys())
    mle_means = [results_by_type[ft]["mle_mean"] for ft in ft_names]
    post_means = [results_by_type[ft]["posterior_mean"] for ft in ft_names]
    ci_lowers = [results_by_type[ft]["ci_lower"] for ft in ft_names]
    ci_uppers = [results_by_type[ft]["ci_upper"] for ft in ft_names]

    fig2 = go.Figure()
    all_vals = mle_means + post_means
    line_min, line_max = min(all_vals) * 0.95, max(all_vals) * 1.05
    fig2.add_trace(
        go.Scatter(
            x=[line_min, line_max],
            y=[line_min, line_max],
            mode="lines",
            name="y=x (no shrinkage)",
            line={"color": "gray", "dash": "dash"},
        )
    )
    fig2.add_trace(
        go.Scatter(
            x=mle_means,
            y=post_means,
            mode="markers+text",
            text=ft_names,
            textposition="top center",
            name="Flat types",
            marker={"color": "red", "size": 10},
            error_y={
                "type": "data",
                "symmetric": False,
                "array": [u - m for u, m in zip(ci_uppers, post_means)],
                "arrayminus": [m - lo for m, lo in zip(post_means, ci_lowers)],
            },
        )
    )
    fig2.update_layout(
        title="Bayesian Regularisation: MLE vs Posterior Mean by Flat Type",
        xaxis_title="MLE Mean ($)",
        yaxis_title="Posterior Mean ($)",
        height=500,
    )
    fig2.write_html(str(OUTPUT_DIR / "flat_type_comparison.html"))
    print("Saved: flat_type_comparison.html")



### Checkpoint 3


In [ ]:
running_means = np.cumsum(bootstrap_means) / np.arange(1, n_bootstrap + 1)
assert (
    abs(running_means[-1] - mle.mean) < mle.standard_error * 3
), "Bootstrap mean should converge to MLE mean"
for ft, r in results_by_type.items():
    assert r["posterior_std"] < sigma_0, f"{ft}: posterior std should shrink from prior"
    assert 0 < r["prior_weight"] < 100, f"{ft}: prior weight must be between 0-100%"
print("\n✓ Checkpoint 3 passed — visualisations and flat-type posteriors valid\n")



## TASK 5 — APPLY: Business Interpretation Synthesis


1. MARKET POSITION: The average 4-room HDB resale price is {fmt_money(mle.mean)}
   with a standard error of {fmt_money(mle.standard_error)}. This estimate is very
   precise — based on {mle.n:,} recent transactions.

2. BAYESIAN ESTIMATE: Even starting with a prior belief of {fmt_money(mu_0)},
   the data overwhelms the prior — the posterior mean is
   {fmt_money(posterior_full.mean)}. Prior assumptions barely matter with this
   volume of data. For the fund manager, this is reassuring.

3. FLAT TYPE VARIATION: Prior influence varies dramatically by segment.
   For abundant flat types (4-room), the prior contributes <0.01% —
   pure data-driven. For rare types (2-room, Executive), the prior
   contributes more, meaning market assumptions play a larger role.

4. INTERVAL AGREEMENT: All four interval methods (Normal, Bootstrap,
   BCa, Bayesian) agree closely with n={mle.n:,}. For smaller segments
   or skewed distributions, BCa bootstrap is recommended.

5. SAMPLING BIAS WARNING: The friendship paradox applies to property
   markets — properties that appear in many listings are not
   representative. High-visibility properties skew perception upward.

6. PORTFOLIO RECOMMENDATION: With {fmt_money(expected_price)} as the
   transaction-weighted expected price, set listing floors per segment
   using the posterior credible intervals, not point estimates.


In [ ]:
# Synthesize all findings from Exercises 1.1–1.4 into a stakeholder-
# ready summary.

print("=== APPLICATION: Property Valuation Insights for Fund Manager ===")
print(
)



### Checkpoint 4


In [ ]:
print("✓ Checkpoint 4 passed — business interpretation complete\n")



## REFLECTION


✓ Credible vs confidence interval: philosophical and practical
    differences validated with 1,000 simulations
  ✓ Bootstrap CI (percentile + BCa): non-parametric alternatives
    that relax distributional assumptions
  ✓ Expected value: transaction-weighted average differs from simple
    average because of volume imbalance across flat types
  ✓ Sampling bias: the friendship paradox shows why naive averages
    systematically mislead
  ✓ Bayesian regularisation across flat types: less data → prior
    matters more → estimates shrink toward the prior
  ✓ Business synthesis: translating statistical results into
    actionable portfolio decisions for a fund manager

  EXERCISE 1 COMPLETE. In Exercise 2, you'll implement MLE from
  scratch using scipy.optimize.minimize, add a prior to get MAP
  estimation, explore the Central Limit Theorem through simulation,
  and diagnose three cases where MLE fails (small n, multimodal
  data, misspecified likelihood). Singapore GDP growth data.


In [ ]:
print("═" * 70)
print("  WHAT YOU'VE MASTERED (1.4 — Intervals & Comparison)")
print("═" * 70)
print(
)

print("\n✓ Exercise 1.4 complete — Intervals and Flat-Type Comparison")
print("✓ EXERCISE 1 COMPLETE — Probability and Bayesian Thinking")

